处理22版工商数据：合并主笔和附表的数据，筛选医院坐标数据

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import time, sqlalchemy

In [2]:
user = "company"    # user id
key = "wisesoe123"    # key
address = "172.16.8.186"
database = "company"

path = user + ":" + key + "@" + address + ":3306/" + database
engine = sqlalchemy.create_engine("mysql+pymysql://" + path)


In [3]:
# 工商-企业主表
sql = text("SELECT * FROM company_basic")

company_basic = pd.read_sql(sql, engine)

In [4]:
print("company_basic shape:", company_basic.shape)

company_basic shape: (86791851, 27)


In [5]:
company_basic.head()

,companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,...,OPFROM,OPTO,OPSCOPE,DOM,DOMDISTRICT,NIC_ID,CANDATE,REVDATE,created,updated
0,16,,北京晨鑫旭日服装服饰设计有限公司,91110115802882789F,110115007262820,802882789,王同兴,,1151,1998-09-21,...,1998-09-21,None,服装设计；销售服装、服饰、针纺织品、服装辅料。（企业依法自主选择经营项目，开展经营活动；依法...,北京市大兴区金星西路6号院1号楼2层（02）202-1-55,110115,C181,None,None,2022-01-18 16:24:37,2022-06-04 13:38:54
1,32,PROVINCENODENUM110000FC70584217944A5ABD65822C7...,北京市琼花电器成套设备销售中心,91110115101970380K,110115004718332,101970380,王生正,,3200,1993-08-16,...,1993-08-16,None,零售高低压电器成套设备，刀开关、刀熔开关系列及安装、调试，零售PVC塑料胶管、机械电器设备。...,北京市大兴区魏善庄镇北田各庄村中街西十条叁号,110115,F5175,None,None,2021-07-07 11:02:13,2022-04-10 07:48:53
2,48,PROVINCENODENUM1100003B6C615988304CEC9DEB8D087...,北京渗不湿防水防腐工程技术有限公司,91110101801234173J,110101003283806,801234173,王夏爱,,1130,2001-09-17,...,2001-09-17,2031-09-16,防水防腐材料的技术开发、服务、转让；销售建筑材料、五金交电、化工（不含危险化学品）。（不含化...,北京市东城区朝阳门内大街199号,110101,M759,None,2019-11-18,2021-05-20 18:21:10,2022-08-31 17:19:44
3,64,PROVINCENODENUM11000085C504CE7FDC4F29898F160C8...,北京国际招标有限公司,91110101400614714Q,110101000995985,400614714,郭凤鸣,,1190,1999-12-21,...,2000-12-11,2050-12-10,招投标代理服务；采购代理服务；政府采购代理服务；工程管理服务；科技中介服务；工程造价咨询业务...,北京市东城区朝阳门北小街71号,110101,F5189,None,None,2021-07-07 11:02:13,2022-09-27 15:13:04
4,80,PROVINCENODENUM11000045D30A4254734CF4B62566633...,北京超市发连锁股份有限公司超市发甘家口店,91110108746112201B,110108004555951,746112201,赵萌,负责人,2229,2002-09-06,...,2002-09-06,None,销售日用杂品、食用农产品、针纺织品、工艺美术品、医疗器械II类、五金交电、家具（不得从事实体...,北京市海淀区甘家口13号,110108,F5219,None,None,2021-07-07 11:02:13,2022-07-21 16:20:19


In [5]:
# 工商-企业主表-附加字段
sql_plus = text("""
    SELECT companyid, lat, lng
    FROM company_basic_plus
    WHERE lat IS NOT NULL AND lng IS NOT NULL
""")

company_plus = pd.read_sql(sql_plus, engine)

print("company_basic_plus shape:", company_plus.shape)

company_basic_plus shape: (81609403, 3)


In [8]:
company_plus.head()

,companyid,lat,lng
0,1,40.465550,115.997140
1,2,40.373561,116.847104
2,3,39.858520,116.801410
3,4,39.863680,116.508620
4,5,39.892950,116.270550


In [9]:
# companyid 类型统一
company_basic["companyid"] = company_basic["companyid"].astype(str)
company_plus["companyid"]  = company_plus["companyid"].astype(str)

print("plus 重复 companyid 行数：",
      company_plus["companyid"].duplicated().sum())



plus 重复 companyid 行数： 0


In [10]:
#合并主表和附表数据
company_merged = pd.merge(
    company_basic,
    company_plus,
    on="companyid",
    how="left"
)

print("merged shape:", company_merged.shape)


merged shape: (86791851, 29)


In [11]:
# 坐标缺失情况
company_merged[["lat", "lng"]].isna().sum()


lat    5207852
lng    5207852
dtype: int64

In [12]:
company_merged.head()


,companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,...,OPSCOPE,DOM,DOMDISTRICT,NIC_ID,CANDATE,REVDATE,created,updated,lat,lng
0,16,,北京晨鑫旭日服装服饰设计有限公司,91110115802882789F,110115007262820,802882789,王同兴,,1151,1998-09-21,...,服装设计；销售服装、服饰、针纺织品、服装辅料。（企业依法自主选择经营项目，开展经营活动；依法...,北京市大兴区金星西路6号院1号楼2层（02）202-1-55,110115,C181,None,None,2022-01-18 16:24:37,2022-06-04 13:38:54,39.83853,116.406860
1,32,PROVINCENODENUM110000FC70584217944A5ABD65822C7...,北京市琼花电器成套设备销售中心,91110115101970380K,110115004718332,101970380,王生正,,3200,1993-08-16,...,零售高低压电器成套设备，刀开关、刀熔开关系列及安装、调试，零售PVC塑料胶管、机械电器设备。...,北京市大兴区魏善庄镇北田各庄村中街西十条叁号,110115,F5175,None,None,2021-07-07 11:02:13,2022-04-10 07:48:53,39.63148,116.414675
2,48,PROVINCENODENUM1100003B6C615988304CEC9DEB8D087...,北京渗不湿防水防腐工程技术有限公司,91110101801234173J,110101003283806,801234173,王夏爱,,1130,2001-09-17,...,防水防腐材料的技术开发、服务、转让；销售建筑材料、五金交电、化工（不含危险化学品）。（不含化...,北京市东城区朝阳门内大街199号,110101,M759,None,2019-11-18,2021-05-20 18:21:10,2022-08-31 17:19:44,39.92522,116.423440
3,64,PROVINCENODENUM11000085C504CE7FDC4F29898F160C8...,北京国际招标有限公司,91110101400614714Q,110101000995985,400614714,郭凤鸣,,1190,1999-12-21,...,招投标代理服务；采购代理服务；政府采购代理服务；工程管理服务；科技中介服务；工程造价咨询业务...,北京市东城区朝阳门北小街71号,110101,F5189,None,None,2021-07-07 11:02:13,2022-09-27 15:13:04,39.92958,116.425880
4,80,PROVINCENODENUM11000045D30A4254734CF4B62566633...,北京超市发连锁股份有限公司超市发甘家口店,91110108746112201B,110108004555951,746112201,赵萌,负责人,2229,2002-09-06,...,销售日用杂品、食用农产品、针纺织品、工艺美术品、医疗器械II类、五金交电、家具（不得从事实体...,北京市海淀区甘家口13号,110108,F5219,None,None,2021-07-07 11:02:13,2022-07-21 16:20:19,39.92653,116.330510


In [13]:
company_merged.to_csv(
    r"Z:\cwx\hospital\company_basic_with_coord_22.csv",
    index=False,
    encoding="utf-8-sig"
)


In [1]:
import pandas as pd
df=pd.read_csv(r"Z:\cwx\hospital\company_basic_with_coord_22.csv")  

C:\Users\chenying\AppData\Local\Temp\13\ipykernel_47516\2311315358.py:2: DtypeWarning: Columns (1,7,8,14,15,21,22,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(r"Z:\cwx\hospital\company_basic_with_coord_22.csv")


In [2]:
print(df["companyid"].dtype)
df["companyid"] = df["companyid"].astype(str)


int64


In [3]:
#关键词筛选（所有医疗机构）

import pandas as pd
import numpy as np
import os

def filter_hospital_data_by_keywords(
    keywords_excel_path,  # 关键词Excel文件路径（第一列是关键词）
    raw_data_csv_path,    # 原始待筛选数据CSV路径
    save_csv_path="./筛选后的医院数据.csv"  # 筛选结果保存路径
):
    """
    核心功能：仅针对ENTNAME(企业名称)和OPSCOPE(经营范围)列，用Excel关键词筛选数据并保存
    """
    # -------------------------- 第一步：读取关键词Excel --------------------------
    try:
        # 仅读取Excel第一列，处理空值和格式问题
        df_keywords = pd.read_excel(keywords_excel_path, usecols=[0])
        first_col_name = df_keywords.columns[0]
        
        # 数据清洗：去空值、转字符串、去首尾空格
        df_keywords = df_keywords.dropna(subset=[first_col_name])
        df_keywords[first_col_name] = df_keywords[first_col_name].astype(str).str.strip()
        
        # 提取去重后的关键词列表
        keywords_list = list(df_keywords[first_col_name].unique())
        print(f"关键词读取完成：共{len(keywords_list)}个不重复关键词")
        print(f"前5个关键词：{keywords_list[:5]}")
        
    except Exception as e:
        print(f"读取关键词Excel失败：{str(e)}")
        print("   检查点：1. Excel文件路径是否正确 2. Excel是否损坏 3. 第一列是否有数据")
        return None
    
    # -------------------------- 第二步：读取原始医院数据 --------------------------
    try:
        # 读取原始CSV数据（适配大文件，低内存模式）
        df_raw = pd.read_csv(
            raw_data_csv_path,
            low_memory=False  # 避免混合数据类型警告
        )
        print(f"原始数据读取完成：共{df_raw.shape[0]}行，{df_raw.shape[1]}列")
        
        # 检查指定列是否存在
        target_columns = ['ENTNAME', 'OPSCOPE']
        missing_cols = [col for col in target_columns if col not in df_raw.columns]
        if missing_cols:
            print(f"原始数据中缺少指定列：{missing_cols}")
            print(f"   数据中实际包含的列：{df_raw.columns.tolist()}")
            return None
        print(f"确认列存在：{target_columns}")
        
    except Exception as e:
        print(f"读取原始CSV数据失败：{str(e)}")
        print("   检查点：1. CSV文件路径是否正确 2. 文件是否被占用 3. 编码是否为UTF-8")
        return None
    
    # -------------------------- 第三步：仅筛选ENTNAME和OPSCOPE列 --------------------------
    try:
        # 定义关键词匹配函数
        def contains_keyword(text):
            """判断文本是否包含任意关键词"""
            if pd.isna(text) or text == '':
                return False
            return any(keyword in str(text) for keyword in keywords_list)
        
        # 分别筛选两列，只要任意一列匹配就保留该行
        mask_entname = df_raw['ENTNAME'].apply(contains_keyword)
        mask_opscope = df_raw['OPSCOPE'].apply(contains_keyword)
        filter_mask = mask_entname | mask_opscope
        
        # 应用筛选，生成新的DataFrame
        df_filtered = df_raw[filter_mask].copy()
        print(f"数据筛选完成：")
        print(f"   - 仅筛选ENTNAME/OPSCOPE列")
        print(f"   - 原始{df_raw.shape[0]}行 → 筛选后{df_filtered.shape[0]}行")
        print(f"   - 企业名称匹配：{mask_entname.sum()}行")
        print(f"   - 经营范围匹配：{mask_opscope.sum()}行")
        
    except Exception as e:
        print(f"数据筛选失败：{str(e)}")
        return None
    
    # -------------------------- 第四步：保存筛选结果 --------------------------
    try:
        # 自动创建保存目录（如果路径包含不存在的文件夹）
        save_dir = os.path.dirname(save_csv_path)
        if save_dir and not os.path.exists(save_dir):
            os.makedirs(save_dir)
            print(f"创建保存目录：{save_dir}")
        
        # 保存CSV（utf-8-sig编码解决中文乱码，index=False不保存行索引）
        df_filtered.to_csv(
            save_csv_path,
            index=False,
            encoding='utf-8-sig'
        )
        print(f"筛选结果已保存：{save_csv_path}")
        
        # 返回筛选后的数据，方便后续分析
        return df_filtered
    
    except Exception as e:
        print(f"保存CSV失败：{str(e)}")
        print("   检查点：1. 保存路径是否有权限 2. 磁盘是否已满 3. 文件名是否合法")
        return df_filtered

# -------------------------- 核心调用 --------------------------
if __name__ == "__main__":
    # 1. 配置文件路径
    KEYWORDS_EXCEL_PATH = r"Z:\cwx\hospital\关键词提取_20251205.xlsx"  
    RAW_DATA_CSV_PATH = r"Z:\cwx\hospital\company_basic_with_coord_22.csv"  
    SAVE_CSV_PATH = r"Z:\cwx\hospital\筛选后的医院数据22.csv"  
    
    # 2. 执行筛选并保存
    df_result = filter_hospital_data_by_keywords(
        keywords_excel_path=KEYWORDS_EXCEL_PATH,
        raw_data_csv_path=RAW_DATA_CSV_PATH,
        save_csv_path=SAVE_CSV_PATH
    )
    
    # 3. 可选：查看筛选结果的企业名称和经营范围列（验证筛选效果）
    if df_result is not None and not df_result.empty:
        print("\n筛选结果预览（仅显示企业名称+经营范围）：")
        print(df_result[['ENTNAME', 'OPSCOPE']].head())

关键词读取完成：共52个不重复关键词
前5个关键词：['医', '卫生', '保健', '康复', '体检']
原始数据读取完成：共86791851行，29列
确认列存在：['ENTNAME', 'OPSCOPE']
数据筛选完成：
   - 仅筛选ENTNAME/OPSCOPE列
   - 原始86791851行 → 筛选后23782116行
   - 企业名称匹配：4477605行
   - 经营范围匹配：23147905行
筛选结果已保存：Z:\cwx\hospital\筛选后的医院数据22.csv

筛选结果预览（仅显示企业名称+经营范围）：
                 ENTNAME                                            OPSCOPE
3             北京国际招标有限公司  招投标代理服务；采购代理服务；政府采购代理服务；工程管理服务；科技中介服务；工程造价咨询业务...
4   北京超市发连锁股份有限公司超市发甘家口店  销售日用杂品、食用农产品、针纺织品、工艺美术品、医疗器械II类、五金交电、家具（不得从事实体...
9         北京正宏嘉名建设工程有限公司  施工总承包、专业承包；园林绿化工程；园林绿化管理；室内外装饰装修、防水工程施工；制作安装弱电...
11         北京普源华燕导热油有限公司  生产桶装导热油；导热油泵、阀门、导热油成套设备工程设计、开发、服务；油品分析检测；销售水质药...
15        北京合义中利医疗设备有限公司  销售医疗器械Ⅱ、Ⅲ类（以《医疗器械经营企业许可证》核定的范围为准）；销售计算机、软件及辅助设...


In [4]:
import csv
import os

file_path = r"Z:\cwx\hospital\筛选后的医院数据22.csv"
output_dir = r"Z:\cwx\hospital\划分后大文件22"

os.makedirs(output_dir, exist_ok=True)

# 第一步：统计总行数
print("正在统计文件总行数...")
with open(file_path, 'r', encoding='utf-8') as f:
    # 使用csv.reader确保正确计数
    reader = csv.reader(f)
    total_rows = sum(1 for _ in reader)

print(f"文件总行数: {total_rows:,}")

# 计算一半的位置
half_point = total_rows // 2  # 注意：这里包括表头

# 第二步：重新读取并写入两个文件
output_file1 = os.path.join(output_dir, "医院数据_前半部分.csv")
output_file2 = os.path.join(output_dir, "医院数据_后半部分.csv")

print("开始划分文件...")
with open(file_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    
    # 读取表头
    header = next(reader)
    
    # 打开两个输出文件
    with open(output_file1, 'w', newline='', encoding='utf-8') as f1, \
         open(output_file2, 'w', newline='', encoding='utf-8') as f2:
        
        writer1 = csv.writer(f1)
        writer2 = csv.writer(f2)
        
        # 写入表头到两个文件
        writer1.writerow(header)
        writer2.writerow(header)
        
        # 逐行处理
        for i, row in enumerate(reader):
            if i < half_point - 1:  # 减1是因为已经读取了表头
                writer1.writerow(row)
            else:
                writer2.writerow(row)
            
            # 显示进度
            if (i + 1) % 1000000 == 0:
                print(f"已处理: {i + 1:,} 行")

print(f"划分完成！")
print(f"前半部分行数: {half_point}")
print(f"后半部分行数: {total_rows - half_point}")
print(f"文件保存到: {output_dir}")

正在统计文件总行数...
文件总行数: 23,782,117
开始划分文件...
已处理: 1,000,000 行
已处理: 2,000,000 行
已处理: 3,000,000 行
已处理: 4,000,000 行
已处理: 5,000,000 行
已处理: 6,000,000 行
已处理: 7,000,000 行
已处理: 8,000,000 行
已处理: 9,000,000 行
已处理: 10,000,000 行
已处理: 11,000,000 行
已处理: 12,000,000 行
已处理: 13,000,000 行
已处理: 14,000,000 行
已处理: 15,000,000 行
已处理: 16,000,000 行
已处理: 17,000,000 行
已处理: 18,000,000 行
已处理: 19,000,000 行
已处理: 20,000,000 行
已处理: 21,000,000 行
已处理: 22,000,000 行
已处理: 23,000,000 行
划分完成！
前半部分行数: 11891058
后半部分行数: 11891059
文件保存到: Z:\cwx\hospital\划分后大文件22


In [3]:
#判断是否存在医院Id


df["companyid"] = df["companyid"].astype(str)


hospital_ids = [
    "902460976",  # "厦门大学医院"
    "920391476",  # "厦门弘爱医院"
    "910562281",  # "北京市朝阳区第三医院"
    "10979696",   # 临沂利民医院
    "12487920"    # 莱芜仁爱医院
]

df_hos = df[df["companyid"].isin(hospital_ids)]

print(f"✅ 数据库中匹配到 {df_hos.shape[0]} 条记录")
display(df_hos)


✅ 数据库中匹配到 5 条记录


,companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,...,OPSCOPE,DOM,DOMDISTRICT,NIC_ID,CANDATE,REVDATE,created,updated,lat,lng
678073,10979696,NaN,临沂利民医院,NaN,371302329021904,NaN,陈玉凤,NaN,4540,2007-07-06,...,内科、外科、耳鼻咽喉科、急诊医学科、医学影像科、医学检验科、妇科。（按许可证核准的有效期限经营）,NaN,371302.0,Q831,NaN,NaN,2021-05-20 18:48:26,2022-09-01 21:51:56,35.148944,118.180299
772335,12487920,PROVINCENODENUM370000371202001012011060229282,莱芜仁爱医院,913701167554015232,371202300004514,755401523,林文东,投资人,4540,2011-06-03,...,内科（呼吸内科专业、消化内科专业）、外科（普通外科专业、泌尿外科专业）、妇产科（妇科专业、生...,莱芜市胜利南路16号,370116.0,Q831,2016-03-18,NaN,2022-01-02 17:05:54,2022-08-14 01:29:22,36.210610,117.663320
5282831,902460976,NaN,厦门大学医院,222494350203310122,222494350203310122,020331012,NaN,NaN,NaN,1921-04-06,...,NaN,NaN,350203,NaN,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
27059927,920391476,NaN,厦门弘爱医院,52350200MJB963084F,厦民证字第020009号,MJB963084,应敏刚,NaN,B52,2016-11-04,...,预防保健科/内科；呼吸内科专业；消化内科专业；神经内科专业；心血管内科专业；肾病学专业；内分...,厦门市湖里区仙岳路3777号,350206.0,NaN,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
54163979,910562281,NaN,北京市朝阳区第三医院,12110105400835357A,NaN,400835357,朱庆生,NaN,B12,2016-12-12,...,为人民身体健康提供医疗与护理保健服务。诊疗与护理,北京市朝阳区双桥南路甲8号（朝阳区第三医院）,110105,NaN,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN


In [4]:
#过滤医院数据：行业代码

# 定义需要筛选的NIC_ID列表
target_nic_ids = [
    "Q831",
    "Q8311",
    "Q8312",
    "Q8313",
    "Q8314",
    "Q8315"
]

# 统一处理NIC_ID列（转字符串+去空格，避免空值/格式问题）
df['NIC_ID'] = df['NIC_ID'].astype(str).str.strip()

# 组合筛选条件：指定值 OR 空值
filter_condition = (df['NIC_ID'].isin(target_nic_ids)) | (df['NIC_ID'].isin(['nan', '']))

# 应用筛选
df_filtered_nic = df[filter_condition].copy()  


print(f"原始数据总行数: {df.shape[0]}")
print(f"筛选后总行数: {df_filtered_nic.shape[0]}")

原始数据总行数: 86791851
筛选后总行数: 4746568


In [5]:
df_filtered_nic["NIC_ID"].value_counts(dropna=False)

NIC_ID
nan      4726193
Q831       15755
Q8311       2817
Q8312       1073
Q8315        675
Q8313         46
Q8314          9
Name: count, dtype: int64

In [6]:
# 定义需要剔除的ENTTYPE值列表
exclude_enttypes = [
    "B34", "B51", "B53", "B55", "B59",
    "C71", "C81", "C89"
]

# 筛选条件：ENTTYPE不在剔除列表中（~表示取反）
enttype_filter = ~df_filtered_nic['ENTTYPE'].isin(exclude_enttypes)
df_filtered_nic_enttype = df_filtered_nic[enttype_filter].copy()

print(f"   - NIC_ID筛选后行数: {df_filtered_nic.shape[0]}")
print(f"   - 剔除ENTTYPE后最终行数: {df_filtered_nic_enttype.shape[0]}")


   - NIC_ID筛选后行数: 4746568
   - 剔除ENTTYPE后最终行数: 3488152


In [7]:
df_filtered_nic_enttype

,companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,...,OPSCOPE,DOM,DOMDISTRICT,NIC_ID,CANDATE,REVDATE,created,updated,lat,lng
1971,31552,NaN,北京美迦博雅动物医院有限公司,91110108573156731K,110108013824971,573156731,刘树新,NaN,1190,2011-04-29,...,动物诊疗；销售宠物用品、饲料、服装、玩具、日用品；宠物美容服务。（市场主体依法自主选择经营项...,北京市海淀区龙翔路1号底商3号商铺,110108.0,Q831,NaN,NaN,2021-05-20 18:21:47,2022-07-21 13:54:14,39.982043,116.375412
3554,56880,NaN,庞泉沟世界公园集团,NaN,JT14112290000018,NaN,NaN,NaN,8100,2014-01-16,...,NaN,NaN,141122.0,nan,NaN,NaN,2021-05-20 18:22:33,2022-09-01 10:50:08,NaN,NaN
3843,61504,NaN,华美控股集团,NaN,141000000060438A,NaN,NaN,NaN,8100,2011-06-01,...,NaN,NaN,141000.0,nan,NaN,NaN,2021-05-20 18:22:33,2022-09-01 10:52:40,NaN,NaN
8155,130496,PROVINCENODENUM1400002140100000062037,太原山大多收宝经销有限公司,91140100MA7YDU1U5L,1401002007575,MA7YDU1U5,彭根生,NaN,1130,1999-09-02,...,NaN,坞城路36号,140100.0,nan,2022-01-12,NaN,2021-05-20 18:25:07,2022-02-22 02:27:23,37.794000,112.586200
10051,160832,NaN,安徽万联投资集团,NaN,340100000604401,NaN,徐庆祝,NaN,8100,2012-01-06,...,NaN,安徽省合肥市庐阳区,340103.0,nan,NaN,NaN,2021-05-20 18:26:57,2022-09-01 10:56:39,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86791846,940448783,NaN,世界宣明会-中国基金有限公司（香港）江西代表处,G1360000MCW162810U,NaN,MCW162810,伍杰(NGERNESTKIT),NaN,CG1,2017-04-19,...,范畴涵盖留守儿童福利事业发展、志愿服务、未成年人保护、防灾减灾救灾知识宣传和技能培训、款物捐...,江西省赣州市石城县琴江镇南苑路73号民政福利院,360735.0,nan,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
86791847,940448815,NaN,克罗地亚国家旅游局上海代表处,G1310000MCW009277J,NaN,MCW009277,弗兰卡(FrankaGulin),NaN,CG1,2018-12-10,...,从事有关在华旅游及旅游文化相关的贸易和投资方面的信息交流、市场调研，为促进两国间的旅游文化交...,申武路159号虹桥丽宝广场2号楼601B室,310000.0,nan,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
86791848,940448831,NaN,韩国财团法人济州特别自治道中小企业综合支援中心上海代表处,G13100003510806471,NaN,351080647,NaN,NaN,CG1,2017-09-18,...,NaN,上海市延安西路2299号11楼11B74、11B76室,310000.0,nan,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
86791849,940448847,NaN,一般财团法人多媒体振兴中心（日本）北京代表处,G111000076675579X0,NaN,76675579X,小原弘嗣（HirotsuguKOHARA）,NaN,CG1,2018-01-10,...,对中国的信息通信等领域的政策、动向等进行调查研究，和中国相关机构进行合作、配合以及交流等，推...,建国门外大街26号长富宫办公楼3002室,210603.0,nan,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN


In [8]:
# 筛选企业名称中包含“医院”的记录
target_column = 'ENTNAME'
keyword_filter = df_filtered_nic_enttype[target_column].str.contains('医院', na=False)
df_hospitals = df_filtered_nic_enttype[keyword_filter].copy()

# 输出结果
print(f"   - 包含“医院”的记录数: {df_hospitals.shape[0]}")

   - 包含“医院”的记录数: 41015


In [15]:
df_hospitals

,companyid,PRIPID,ENTNAME,UNISCID,REGNO,NACAOID,NAME,NAMETITLE,ENTTYPE,ESDATE,...,OPSCOPE,DOM,DOMDISTRICT,NIC_ID,CANDATE,REVDATE,created,updated,lat,lng
1971,31552,NaN,北京美迦博雅动物医院有限公司,91110108573156731K,110108013824971,573156731,刘树新,NaN,1190,2011-04-29,...,动物诊疗；销售宠物用品、饲料、服装、玩具、日用品；宠物美容服务。（市场主体依法自主选择经营项...,北京市海淀区龙翔路1号底商3号商铺,110108.0,Q831,NaN,NaN,2021-05-20 18:21:47,2022-07-21 13:54:14,39.982043,116.375412
15229,243680,NaN,墨玉华康医院,NaN,653222080000288,NaN,李斌,NaN,4540,2007-05-08,...,西医呼吸内科专业、消化内科专业、心血管内科、内分泌专业、西医普通外科、五官科。\t一般经营项...,墨玉县其乃巴格路24号,653222.0,Q831,NaN,NaN,2022-01-18 16:24:37,2022-09-01 11:00:27,37.277340,79.728841
18786,300592,PROVINCENODENUM520000252000000000067223,贵阳金阳贵钢康复医院,915201156629850568,520115000152180,662985056,胡文卓,负责人,4110,2007-08-07,...,法律、法规、国务院决定规定禁止的不得经营；法律、法规、国务院决定规定应当许可（审批）的，经审...,贵州省贵阳市观山湖区金阳新区白云大道311号,520115.0,Q831,NaN,NaN,2021-08-30 18:29:09,2022-06-03 07:54:32,26.632757,106.655211
22064,353040,NaN,阿拉善丽人医院有限责任公司,91152900585153220T,152900000010938,585153220,林清远,NaN,1130,2011-10-26,...,无 一般经营项目：一般经营项目：急诊科、内科、外科、妇科、预防保健科、医学检验、医学影像科,内蒙古自治区阿拉善盟阿拉善左旗巴彦浩特镇和硕特南路,152900.0,Q831,2017-01-05,NaN,2022-01-18 16:24:37,2022-07-17 09:33:16,38.833507,105.668991
23530,376496,NaN,沈阳市爱心女子医院,NaN,210102100007090,NaN,林国营,NaN,4531,2006-03-17,...,以《医疗机构许可证》核定项目为准。,NaN,210102.0,Q831,NaN,NaN,2021-05-20 18:38:25,2022-09-01 11:09:43,41.796842,123.424816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86779591,921265791,NaN,盖州民和中医皮肤病医院,52210881MJ32784022,NaN,MJ3278402,郑兰德,NaN,B52,2018-11-30,...,皮肤病/中医科*******,辽宁省营口市盖州市西城办事处站前社区,210881.0,nan,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
86780975,935279951,NaN,东营垦利区人民医院法医司法鉴定所,12370521493447002M,137052100084,493447002,崔志平,NaN,B12,2002-11-24,...,为人民身体健康提供医疗与护理保健服务。医疗与护理、医学教学、医学研究、医疗人员培训、卫生技术...,东营市垦利区新兴路99号,370505.0,nan,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
86780994,935280543,NaN,绥棱县人民医院,34230000414495189G,231206011,414495189,白雪峰,NaN,B12,2006-01-23,...,为人民身体健康提供医疗与护理保健服务。医疗与护理卫生医疗人员培训卫生技术人员继续教育保健与健康教育,绥棱县繁荣大街381号,231226.0,nan,2019-03-29,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN
86781010,935281023,NaN,南京医科大学第一附属医院,12320000466000838D,NaN,466000838,唐金海,NaN,B12,2001-06-27,...,南京市广州路300号,江苏省事业单位登记管理局,320100.0,nan,NaN,NaN,2022-09-14 11:27:11,2022-09-14 11:27:11,NaN,NaN


In [12]:
import re

# 从已经包含“医院”的数据开始
df_filtered = df_hospitals.copy()

# 去除前后空格
df_filtered["ENTNAME"] = df_filtered["ENTNAME"].astype(str).str.strip()

# =========================
# 建立删除标记
# =========================
drop_mask = False

# ① 包含关键词,不删除“中心”结尾的关键词
exclude_keywords = [
    "动物医院",
    "宠物医院",
    "兽医院",
    "兽医医院",
    "农药"
]

for keyword in exclude_keywords:
    drop_mask |= df_filtered["ENTNAME"].str.contains(
        keyword,
        na=False
    )

# ② 结尾关键词
exclude_suffixes = [
    "小卖部",
    "经营部",
    "服务社",
    "工会",
    "委员会",
    "幼儿园",
    "印刷厂",
    "研究所",
    "鉴定所",
    "服务部",
    "服务站"
]

suffix_pattern = "(" + "|".join(map(re.escape, exclude_suffixes)) + r")$"

drop_mask |= df_filtered["ENTNAME"].str.contains(
    suffix_pattern,
    regex=True,
    na=False
)

# =========================
# 保存删除的数据
# =========================
deleted = df_filtered.loc[drop_mask].copy()

# 保留医院
df_filtered = df_filtered.loc[~drop_mask].copy()

print(f"删除 {len(deleted)} 条")
print(f"保留 {len(df_filtered)} 条")

删除 677 条
保留 40338 条


C:\Users\chenying\AppData\Local\Temp\13\ipykernel_47516\3012273308.py:46: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  drop_mask |= df_filtered["ENTNAME"].str.contains(


In [13]:
#查看删除的数据
print(
    deleted["ENTNAME"]
    .drop_duplicates()
    .sort_values()
    .to_string(index=False)
)

                                            三河市兽医院
                                        三门峡市陕州区兽医院
                                      上海安辰宠物医院有限公司
                                      上海盛泰宠物医院有限公司
                                            东丰县兽医院
                                 东丰县吉丰庄稼医院大阳茂源农药商店
                                            东光县兽医院
                                         东港市黑沟镇兽医院
                                         东营中心医院幼儿园
                                    东营利津县中心医院司法鉴定所
                                  东营垦利区人民医院法医司法鉴定所
                                    东营市合德医院肿瘤防治研究所
                                        东营市垦利区动物医院
                                  东营广饶县人民医院法医司法鉴定所
                                 中国人民解放军二二三医院军人服务社
                                中国人民解放军空军北京医院军人服务社
                                中国人民解放军第一五四医院长城幼儿园
                                中国人民解放军第三零二医院军人服务社
                                中国医科大学附属第一医院鞍山医院工会
                          中国教育工

In [16]:

# 1. 剔除名称以"中心"结尾的记录（对应Stata: drop if regexm(entname, "中心$")）
df_filtered = df_hospitals[~df_hospitals['ENTNAME'].str.endswith("中心", na=False)].copy()

# 定义需要排除的关键词列表
exclude_keywords = [
    " ", "卫生", "医务室", "防治", "保健中心", "服务中心", "急救中心",
    "体检中心", "康复中心", "管理中心", "培训中心", "中医馆", "门诊",
    "诊所", "动物", "宠物", "兽医", "农药", "经营部", "工会", "委员会",
    "幼儿园", "服务社"
]

# 2. 剔除名称包含上述任意关键词的记录（对应Stata的多个keep if strpos(...) == 0）
for keyword in exclude_keywords:
    df_filtered = df_filtered[~df_filtered['ENTNAME'].str.contains(keyword, na=False)].copy()

# 输出筛选后的行数
print(f"   - 去除非医院机构后最终行数: {df_filtered.shape[0]}")

   - 去除非医院机构后最终行数: 39347


In [10]:

df_filtered.to_csv(
    r"Z:\cwx\hospital\hospital_2022.csv",
    index=False,
    encoding="utf-8-sig"
)
